In [1]:
import os
import re

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import anndata as ad

%load_ext autoreload
%autoreload 2

In [2]:
ad.__version__

'0.10.9'

## Load data

Download the Visium data from https://zenodo.org/records/6578047.
Download all files whose names start with "Visium" and place them in the ```processed``` folder within the user-specified working directory (```WD```).

In [3]:
# WD = "/data/visium_heart"  # change as needed
WD = "/dss/dssfs02/lwp-dss-0001/pn36po/pn36po-dss-0001/di93vel/visium_heart"

In [4]:
input_dir = os.path.join(WD, "processed")

# Find all .h5ad files in the directory
h5ad_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith(".h5ad")]

# Load each file into an AnnData object
adatas = [sc.read_h5ad(f) for f in h5ad_files]

# Extract clean sample names for each file using regex
sample_names = [re.search(r'Visium_(.+?)\.h5ad', os.path.basename(f)).group(1) for f in h5ad_files]

# Concatenate all AnnData objects
# Adjust `join='outer'` or `join='inner'` depending on whether you want to keep all or only common genes
adata_combined = ad.AnnData.concatenate(*adatas, join='inner', batch_key='sample', batch_categories=sample_names)
adata_combined

/tmp/ipykernel_263833/1306160217.py:14: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_combined = ad.AnnData.concatenate(*adatas, join='inner', batch_key='sample', batch_categories=sample_names)


AnnData object with n_obs × n_vars = 88704 × 11681
    obs: 'n_counts', 'n_genes', 'percent.mt', 'Adipocyte', 'Cardiomyocyte', 'Endothelial', 'Fibroblast', 'Lymphoid', 'Mast', 'Myeloid', 'Neuronal', 'Pericyte', 'Cycling.cells', 'vSMCs', 'cell_type_original', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'sample'
    var: 'features'
    obsm: 'X_pca', 'X_spatial', 'X_umap'

## Inspect data

In [5]:
print(np.min(adata_combined.X))
print(np.max(adata_combined.X))

0.0
8.193758066884202


In [6]:
# Count in how many cells each gene is expressed
gene_expression_counts = np.array((adata_combined.X > 0).sum(axis=0)).flatten()
min_cells_per_gene = gene_expression_counts.min()
print(f"Minimum number of cells a gene is expressed in: {min_cells_per_gene}")

Minimum number of cells a gene is expressed in: 786


In [7]:
# Count how many genes are expressed in each cell
cell_expression_counts = np.array((adata_combined.X > 0).sum(axis=1)).flatten()
min_genes_per_cell = cell_expression_counts.min()
print(f"Minimum number of genes a cell expresses: {min_genes_per_cell}")

Minimum number of genes a cell expresses: 292


In [8]:
adata_combined.obs['sample'].value_counts()

sample
RZ_BZ_P3         4659
GT_IZ_P9         4361
control_P1       4269
FZ_GT_P4         4253
IZ_BZ_P2         4203
GT_IZ_P9_rep2    4113
IZ_P3            3771
IZ_P10           3646
RZ_P9            3626
GT_IZ_P15        3572
RZ_GT_P2         3538
RZ_P6            3484
RZ_BZ_P12        3392
RZ_BZ_P2         3373
FZ_P14           3175
FZ_GT_P19        3100
IZ_P15           3083
RZ_FZ_P5         3082
RZ_P3            2994
control_P7       2931
IZ_P16           2713
FZ_P18           2551
control_P8       2456
FZ_P20           2410
control_P17      2043
RZ_P11           2016
GT_IZ_P13        1890
Name: count, dtype: int64

## Continue building adata

In [9]:
assay_map = {
    'EFO:0010961': 'Visium Spatial Gene Expression'
}

cell_type_map = {
    'CL:0000513': 'cardiac muscle myoblast',
    'CL:0002548': 'fibroblast of cardiac tissue',
    'CL:0010008': 'cardiac endothelial cell',
    'CL:0001082': 'immature innate lymphoid cell',
    'CL:0000003': 'native cell',
    'CL:0000514': 'smooth muscle myoblast',
    'CL:0000669': 'pericyte cell',
    'CL:0000838': 'lymphoid lineage restricted progenitor cell',
    'CL:0000006': 'neuronal receptor cell', 
    'CL:0000097': 'mast cell',
    'CL:1000311': 'adipocyte of epicardial fat of left ventricle'
}

development_stage_map = {
    'HsapDv:0000138': '44-year-old human stage', 
    'HsapDv:0000151': '57-year-old human stage',
    'HsapDv:0000146': '52-year-old human stage',
    'HsapDv:0000137': '43-year-old human stage',
    'HsapDv:0000160': '66-year-old human stage',
    'HsapDv:0000168': '74-year-old human stage',
    'HsapDv:0000132': '38-year-old human stage',
    'HsapDv:0000141': '47-year-old human stage',
    'HsapDv:0000134': '40-year-old human stage',
    'HsapDv:0000152': '58-year-old human stage',
    'HsapDv:0000157': '63-year-old human stage',
    'HsapDv:0000149': '55-year-old human stage',
    'HsapDv:0000158': '64-year-old human stage',
    'HsapDv:0000155': '61-year-old human stage',
    'HsapDv:0000154': '60-year-old human stage',
    'HsapDv:0000145': '51-year-old human stage'
}

disease_map = {
    'MONDO:0005068': 'myocardial infarction',
    'PATO:0000461': 'normal'
}

ethnicity_map = {
    'HANCESTRO:0005': 'European'
}

organism_map = {
    'NCBITaxon:9606': 'Homo sapiens'
}

sex_map = {
    'PATO:0000383': 'female',
    'PATO:0000384': 'male'
}

tissue_map = {
    'UBERON:0002084': 'heart left ventricle'
}

In [10]:
adata_combined.obs['assay'] = adata_combined.obs['assay_ontology_term_id'].map(assay_map)
adata_combined.obs['cell_type'] = adata_combined.obs['cell_type_ontology_term_id'].map(cell_type_map)
adata_combined.obs['development_stage'] = adata_combined.obs['development_stage_ontology_term_id'].map(development_stage_map)
adata_combined.obs['disease'] = adata_combined.obs['disease_ontology_term_id'].map(disease_map)
adata_combined.obs['ethnicity'] = adata_combined.obs['ethnicity_ontology_term_id'].map(ethnicity_map)
adata_combined.obs['organism'] = adata_combined.obs['organism_ontology_term_id'].map(organism_map)
adata_combined.obs['sex'] = adata_combined.obs['sex_ontology_term_id'].map(sex_map)
adata_combined.obs['tissue'] = adata_combined.obs['tissue_ontology_term_id'].map(tissue_map)

In [11]:
adata_combined.obs['condition'] = adata_combined.obs['sample'].str.split('_P').str[0].astype('category')
adata_combined.obs[['sample', 'condition']].drop_duplicates().values.tolist()

[['control_P1', 'control'],
 ['IZ_P15', 'IZ'],
 ['RZ_P6', 'RZ'],
 ['RZ_P3', 'RZ'],
 ['IZ_P16', 'IZ'],
 ['control_P17', 'control'],
 ['control_P7', 'control'],
 ['RZ_P11', 'RZ'],
 ['GT_IZ_P13', 'GT_IZ'],
 ['RZ_BZ_P2', 'RZ_BZ'],
 ['GT_IZ_P9_rep2', 'GT_IZ'],
 ['IZ_P10', 'IZ'],
 ['control_P8', 'control'],
 ['FZ_P14', 'FZ'],
 ['GT_IZ_P15', 'GT_IZ'],
 ['FZ_P20', 'FZ'],
 ['RZ_GT_P2', 'RZ_GT'],
 ['GT_IZ_P9', 'GT_IZ'],
 ['RZ_BZ_P12', 'RZ_BZ'],
 ['IZ_BZ_P2', 'IZ_BZ'],
 ['IZ_P3', 'IZ'],
 ['RZ_BZ_P3', 'RZ_BZ'],
 ['RZ_FZ_P5', 'RZ_FZ'],
 ['RZ_P9', 'RZ'],
 ['FZ_P18', 'FZ'],
 ['FZ_GT_P19', 'FZ_GT'],
 ['FZ_GT_P4', 'FZ_GT']]

In [12]:
adata_combined.obsm['spatial'] = adata_combined.obsm['X_spatial'].copy()

In [13]:
adata_combined.write_h5ad(os.path.join(WD, "adata_combined.h5ad"))

Download the Visium metadata from https://zenodo.org/records/6580069 and place the file ```metadata-Visium.csv``` in the working directory (```WD```).

In [14]:
metadata = pd.read_csv(os.path.join(WD, 'metadata-Visium.csv'))

# Note that no processed data is available for the hca_sample_id ACH0021.
metadata = metadata[metadata['hca_sample_id'] != 'ACH0021']
metadata = metadata.drop_duplicates()
metadata

,patient,patient_region_id,patient_group,major_labl,batch,hca_sample_id
0,P5,RZ/FZ_P5,group_1,FZ,10X,10X0027
1,P3,IZ_P3,group_2,IZ,10X,10X0017
2,P3,RZ/BZ_P3,group_1,BZ,10X,10X0026
3,P3,RZ_P3,group_1,RZ,10X,10X0020
4,P2,IZ/BZ_P2,group_2,IZ,10X,10X0018
5,P2,RZ/BZ_P2,group_1,BZ,10X,10X0025
6,P4,FZ/GT_P4,group_3,FZ,10X,10X009
7,P1,control_P1,group_1,CTRL,10X,10X001
8,P17,control_P17,group_1,CTRL,ACH,ACH002
9,P12,RZ/BZ_P12,group_1,BZ,ACH,ACH0024


In [15]:
# Replace '/' with '_' in patient_region_id to match the sample naming convention in adata_combined.obs
metadata['patient_region_id'] = metadata['patient_region_id'].str.replace('/', '_', regex=False)
metadata_indexed = metadata.set_index('patient_region_id')

In [16]:
# Match metadata rows to adata_combined.obs by sample name (which corresponds to patient_region_id)
matched_metadata = metadata_indexed.loc[adata_combined.obs['sample']]

# Add 'patient' and 'batch' columns from metadata to adata_combined.obs, preserving spot order
adata_combined.obs['patient'] = matched_metadata['patient'].values
adata_combined.obs['batch'] = matched_metadata['batch'].values
adata_combined.obs['hca_sample_id'] = matched_metadata['hca_sample_id'].values
adata_combined.obs

,n_counts,n_genes,percent.mt,Adipocyte,Cardiomyocyte,Endothelial,Fibroblast,Lymphoid,Mast,Myeloid,...,development_stage,disease,ethnicity,organism,sex,tissue,condition,patient,batch,hca_sample_id
AAACAAGTATCTCCCA-1-control_P1,4429.0,2147,29.575465,0.001598,0.437324,0.024356,0.318303,0.022537,0.003858,0.059224,...,44-year-old human stage,normal,European,Homo sapiens,female,heart left ventricle,control,P1,10X,10X001
AAACAATCTACTAGCA-1-control_P1,3037.0,1591,31.299898,0.000482,0.743949,0.082948,0.089687,0.002331,0.000562,0.039163,...,44-year-old human stage,normal,European,Homo sapiens,female,heart left ventricle,control,P1,10X,10X001
AAACACCAATAACTGC-1-control_P1,2507.0,1462,23.956640,0.001974,0.375296,0.167984,0.151615,0.004123,0.033053,0.009395,...,44-year-old human stage,normal,European,Homo sapiens,female,heart left ventricle,control,P1,10X,10X001
AAACAGAGCGACTCCT-1-control_P1,2502.0,1341,33.018868,0.000110,0.652696,0.095914,0.154588,0.005017,0.000977,0.055517,...,44-year-old human stage,normal,European,Homo sapiens,female,heart left ventricle,control,P1,10X,10X001
AAACAGCTTTCAGAAG-1-control_P1,3054.0,1617,33.638132,0.000084,0.533660,0.164157,0.065999,0.004493,0.001532,0.072105,...,44-year-old human stage,normal,European,Homo sapiens,female,heart left ventricle,control,P1,10X,10X001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGTTTCACATCCAGG-1-FZ_GT_P4,1221.0,832,62.439549,0.005341,0.082269,0.052038,0.511566,0.013043,0.000854,0.260980,...,74-year-old human stage,myocardial infarction,European,Homo sapiens,female,heart left ventricle,FZ_GT,P4,10X,10X009
TTGTTTCATTAGTCTA-1-FZ_GT_P4,1001.0,653,56.419458,0.038269,0.344302,0.082631,0.349132,0.010914,0.000517,0.014096,...,74-year-old human stage,myocardial infarction,European,Homo sapiens,female,heart left ventricle,FZ_GT,P4,10X,10X009
TTGTTTCCATACAACT-1-FZ_GT_P4,823.0,588,66.151013,0.000132,0.111444,0.287319,0.276820,0.044779,0.001396,0.184154,...,74-year-old human stage,myocardial infarction,European,Homo sapiens,female,heart left ventricle,FZ_GT,P4,10X,10X009
TTGTTTGTATTACACG-1-FZ_GT_P4,511.0,391,73.827947,0.000386,0.002104,0.090331,0.750387,0.002042,0.000472,0.003078,...,74-year-old human stage,myocardial infarction,European,Homo sapiens,female,heart left ventricle,FZ_GT,P4,10X,10X009


## Save adata

In [17]:
adata_combined.write_h5ad(os.path.join(WD, "adata_annotated.h5ad"))